# Bot detection challenge

Предсказываем вероятность того, что кука — бот, по её событиям за суточное окно.
Метрика — Precision при Recall ≥ 0.70 (`metric.py`).

1. Загрузка, EDA, дубликаты
2. Окно наблюдения и утечка через `captcha_shown`
3. Признаки
4. Валидация и выбор модели
5. Важность признаков
6. Сабмит
7. Описание подхода

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

print("python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)
print("sklearn :", sklearn.__version__)

python  : 3.14.4
numpy   : 2.5.1
pandas  : 3.0.3
sklearn : 1.9.0


In [2]:
train = pd.read_csv(
    "data/train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    "data/test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

print("train:", train.shape, " test:", test.shape, " events:", events.shape)
print("доля ботов в train:", round(train.target.mean(), 4))
print(train.head(3).to_string())

train: (11091, 5)  test: (4909, 4)  events: (328905, 14)
доля ботов в train: 0.0811
             cookie_id   cookie_created_at window_start_ts window_end_ts  target
0  ck_54a059eb7d3ea68b 2025-11-21 09:30:41      2026-04-06    2026-04-07       0
1  ck_7e4de46eeab82974 2025-09-23 10:10:24      2026-04-06    2026-04-07       0
2  ck_9320229ef6304522 2026-03-04 00:08:02      2026-04-06    2026-04-07       0


## 1. EDA

Типы событий, платформы (регистр разный — нужна нормализация), пропуски, окна train/test.

In [3]:
print(events.event_name.value_counts(), "\n")
print(events.platform.value_counts(), "\n")
print("пропуски по колонкам:")
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


In [4]:
print("окна train:", train.window_start_ts.min(), "..", train.window_start_ts.max())
print("окна test :", test.window_start_ts.min(), "..", test.window_start_ts.max())

окна train: 2026-04-06 00:00:00 .. 2026-04-19 00:00:00
окна test : 2026-04-20 00:00:00 .. 2026-04-26 00:00:00


Тест идёт строго после трейна по времени, поэтому валидироваться будем тоже по
времени, а не случайным сплитом.

### Дубликаты

В событиях есть полностью одинаковые строки — похоже на дубли записи в лог. Они
завышают счётчики и дают нулевые интервалы между событиями, поэтому удаляем их.

In [5]:
dup_mask = events.duplicated()
print("полных дублей строк во всех events:", dup_mask.sum(), "из", len(events))
print("затронутых cookie_id:", events.loc[dup_mask, "cookie_id"].nunique())

events = events.drop_duplicates().reset_index(drop=True)
print("events после dedup:", events.shape)

полных дублей строк во всех events: 4863 из 328905
затронутых cookie_id: 3775
events после dedup: (324042, 14)


## 2. Окно наблюдения

Признаки должны быть доступны на конец окна, поэтому берём только события с
`window_start_ts <= event_ts < window_end_ts`. Отдельно проверим капчу — кажется,
что это хороший признак бота.

In [6]:
m = events.merge(
    train[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id"
)
captcha = m[m.eid == 900]

print("captcha_shown у train-куки всего      :", len(captcha))
print(
    "  до начала окна                       :",
    (captcha.event_ts < captcha.window_start_ts).sum(),
)
print(
    "  внутри окна                          :",
    (
        (captcha.event_ts >= captcha.window_start_ts)
        & (captcha.event_ts < captcha.window_end_ts)
    ).sum(),
)
print(
    "  после конца окна                     :",
    (captcha.event_ts >= captcha.window_end_ts).sum(),
)
print(
    "доля ботов среди куки с captcha_shown  :",
    round(
        train.set_index("cookie_id").loc[captcha.cookie_id.unique(), "target"].mean(), 3
    ),
    " (базовая доля:",
    round(train.target.mean(), 3),
    ")",
)

captcha_shown у train-куки всего      : 7831
  до начала окна                       : 0
  внутри окна                          : 0
  после конца окна                     : 7831
доля ботов среди куки с captcha_shown  : 0.771  (базовая доля: 0.081 )


Капча ни разу не попадает в окно — она всегда после его конца, и у 77% таких
кук target = 1 (при 8% в среднем). То есть это реакция антибота уже после
окна — информация из будущего. Не используем.

In [7]:
def filter_window(events, meta):
    # события строго внутри [window_start_ts, window_end_ts)
    ev = events.merge(
        meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id"
    )
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]


ev_tr = filter_window(events, train)
ev_te = filter_window(events, test)
print("событий в окне: train =", len(ev_tr), " test =", len(ev_te))
print(
    "доля событий train-куки вне окна:",
    round(1 - len(ev_tr) / events.cookie_id.isin(train.cookie_id).sum(), 3),
)

событий в окне: train = 195484  test = 88349
доля событий train-куки вне окна: 0.171


## 3. Признаки

Идея — описать не только объём активности, но и её характер: ритм, разнообразие,
технические следы.

- **Объём и разнообразие**: число событий и их типов, уникальные объявления /
  категории / локации / запросы, глубина выдачи, счётчики по типам событий и доли.
- **Повторы**: `item_repeat_rate` — повторные просмотры карточки,
  `item_revisit_frac` — доля событий по уже виденным объявлениям.
- **Ритм**: статистики интервалов между событиями, длительность, темп.
- **Технические следы**: число платформ и user-agent у куки, ключевые слова
  автоматизации в user-agent.
- **Курсор**: разброс `pointer_x`.
- **Возраст куки** на начало окна.

Пробовал и убрал: флаги `has_*` (дублируют счётчики событий), энтропию и число
часов активности, долю pro-продавцов, долю событий с курсором, разброс `pointer_y`,
отдельные флаги на каждое слово user-agent — на валидации пользы от них не было.

In [8]:
EVENT_IDS = [100, 200, 210, 220, 300, 301, 303, 400, 500]
# ключевые слова автоматизации, встречающиеся в user_agent
BOT_UA_PATTERN = "headless|python|curl|scrapy|go-http|okhttp|requests"


def build_features(events_win, meta):
    ev = events_win.sort_values("event_ts").copy()
    ev["platform_norm"] = ev["platform"].str.lower().replace({"iphone": "ios"})
    ev["ua_bot_kw"] = ev["user_agent"].str.lower().str.contains(BOT_UA_PATTERN)

    rows = []
    for cid, g in ev.groupby("cookie_id"):
        n = len(g)
        ts = g["event_ts"].values
        if n >= 2:
            iat = np.diff(ts).astype("timedelta64[s]").astype(float)
            iat_mean, iat_std = iat.mean(), iat.std()
            iat_cv = iat_std / (iat_mean + 1e-6)
            iat_min, iat_median = iat.min(), float(np.median(iat))
            span_sec = (ts[-1] - ts[0]) / np.timedelta64(1, "s")
        else:
            # одно событие — интервалов нет
            iat_mean = iat_std = iat_cv = iat_min = iat_median = np.nan
            span_sec = 0.0

        # доля событий по уже виденным объявлениям
        item_ids = g["item_id"].dropna()
        item_revisit_frac = (
            1 - item_ids.nunique() / len(item_ids) if len(item_ids) else np.nan
        )

        row = {
            "cookie_id": cid,
            "n_events": n,
            "n_eid_distinct": g["eid"].nunique(),
            "item_nunique": g["item_id"].nunique(),
            "item_view_nunique": g.loc[g["eid"] == 200, "item_id"].nunique(),
            "category_nunique": g["item_category"].nunique(),
            "location_nunique": g["item_location"].nunique(),
            "search_query_nunique": g["search_query"].nunique(),
            "search_page_max": g["search_page"].max(),
            "search_page_mean": g["search_page"].mean(),
            "platform_nunique": g["platform_norm"].nunique(),
            "ua_nunique": g["user_agent"].nunique(),
            "ua_bot_kw_any": float(g["ua_bot_kw"].any()),
            "iat_mean": iat_mean,
            "iat_std": iat_std,
            "iat_cv": iat_cv,
            "iat_min": iat_min,
            "iat_median": iat_median,
            "span_sec": span_sec,
            "events_per_min_active": n / (span_sec / 60 + 1),
            "pointer_x_std": g["pointer_x"].std(),  # NaN, если курсора нет совсем
            "item_revisit_frac": item_revisit_frac,
        }
        row.update({f"cnt_eid_{e}": c for e, c in g["eid"].value_counts().items()})
        rows.append(row)

    feat = meta[["cookie_id", "cookie_created_at", "window_start_ts"]].merge(
        pd.DataFrame(rows), on="cookie_id", how="left"
    )
    cnt_cols = [f"cnt_eid_{e}" for e in EVENT_IDS]
    feat[cnt_cols] = feat.reindex(columns=cnt_cols).fillna(
        0
    )  # тип события не встречался -> 0

    feat["item_view_ratio"] = feat["cnt_eid_200"] / feat["n_events"]
    feat["search_ratio"] = feat["cnt_eid_100"] / feat["n_events"]
    feat["photo_swipe_per_item"] = feat["cnt_eid_210"] / feat["item_nunique"].replace(
        0, np.nan
    )
    # повторные просмотры карточки (уникальные id — только среди item_view)
    feat["item_repeat_rate"] = 1 - feat["item_view_nunique"] / feat[
        "cnt_eid_200"
    ].replace(0, np.nan)
    feat["account_age_days"] = (
        feat["window_start_ts"] - feat["cookie_created_at"]
    ).dt.total_seconds() / 86400
    return feat.drop(columns=["cookie_created_at", "window_start_ts"])


Xtr = build_features(ev_tr, train)
Xte = build_features(ev_te, test)
ytr = train.set_index("cookie_id").loc[Xtr.cookie_id, "target"].values

FEATURES = [c for c in Xtr.columns if c != "cookie_id"]
print("число признаков:", len(FEATURES))
print(Xtr.shape, Xte.shape)
print(Xtr[FEATURES].isna().mean().sort_values(ascending=False).head(5))

число признаков: 35
(11091, 36) (4909, 36)
pointer_x_std       0.646380
search_page_mean    0.083040
search_page_max     0.083040
item_repeat_rate    0.065729
iat_mean            0.022901
dtype: float64


Пропуски (нет курсора, одно событие в окне) оставляю как NaN, а не 0: «нет
данных» — это не «ноль». `HistGradientBoostingClassifier` умеет работать с NaN сам.

## 4. Валидация

Обучаемся на ранних окнах, проверяем на поздних; три точки отсечения, метрика из
`metric.py`.

In [9]:
from metric import precision_at_recall
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

window_start = train.set_index("cookie_id").loc[Xtr.cookie_id, "window_start_ts"]
CUTS = ["2026-04-13", "2026-04-15", "2026-04-17"]
BASELINE_COLS = ["n_events", "item_nunique"]

for cut in CUTS:
    is_valid = window_start.ge(cut).values
    print(
        f"cut {cut}: train={(~is_valid).sum()}, valid={is_valid.sum()} (ботов в valid: {ytr[is_valid].sum()})"
    )


def time_cv(make_model, cols):
    # обучение на окнах до cut, проверка на окнах >= cut
    scores = []
    for cut in CUTS:
        is_valid = window_start.ge(cut).values
        model = make_model().fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
        scores.append(
            precision_at_recall(
                ytr[is_valid], model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]
            )
        )
    return np.array(scores)


def make_hgb(**params):
    # без ранней остановки: иначе на всём train (>10k строк) sklearn включит её сам
    return HistGradientBoostingClassifier(
        early_stopping=False, random_state=RANDOM_STATE, **params
    )


def with_imputer(*steps):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), *steps)


CANDIDATES = {
    "baseline (RF, 2 признака quickstart)": (
        lambda: RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE
        ),
        BASELINE_COLS,
    ),
    "LogisticRegression, все признаки": (
        lambda: with_imputer(StandardScaler(), LogisticRegression(max_iter=2000)),
        FEATURES,
    ),
    "RandomForest, все признаки": (
        lambda: with_imputer(
            RandomForestClassifier(
                n_estimators=500,
                min_samples_leaf=3,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )
        ),
        FEATURES,
    ),
    "HistGradientBoosting, все признаки": (
        lambda: make_hgb(
            max_depth=3,
            learning_rate=0.05,
            max_iter=300,
            min_samples_leaf=20,
            l2_regularization=1.0,
        ),
        FEATURES,
    ),
}

print()
print(f"{'модель':38s}" + "".join(f"{c:>12s}" for c in CUTS) + f"{'среднее':>10s}")
for name, (make_model, cols) in CANDIDATES.items():
    s = time_cv(make_model, cols)
    print(f"{name:38s}" + "".join(f"{x:12.4f}" for x in s) + f"{s.mean():10.4f}")

cut 2026-04-13: train=5930, valid=5161 (ботов в valid: 408)
cut 2026-04-15: train=7599, valid=3492 (ботов в valid: 285)
cut 2026-04-17: train=9140, valid=1951 (ботов в valid: 160)

модель                                  2026-04-13  2026-04-15  2026-04-17   среднее
baseline (RF, 2 признака quickstart)        0.0931      0.0912      0.0957    0.0933
LogisticRegression, все признаки            0.4511      0.4435      0.4057    0.4334
RandomForest, все признаки                  0.5468      0.5698      0.7467    0.6211
HistGradientBoosting, все признаки          0.7168      0.7092      0.8175    0.7478


Бустинг сильно лучше baseline из quickstart (~0.75 против ~0.09) и лучше
RandomForest (~0.62) и логрега (~0.43). Разброс между отсечками заметный, поэтому
гиперпараметры выбираю по среднему по трём отсечкам.

In [10]:
CONFIGS = [
    dict(
        max_depth=3,
        learning_rate=0.05,
        max_iter=300,
        min_samples_leaf=20,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=3,
        learning_rate=0.08,
        max_iter=250,
        min_samples_leaf=30,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=4,
        learning_rate=0.05,
        max_iter=400,
        min_samples_leaf=15,
        l2_regularization=1.0,
    ),
    dict(
        max_depth=4,
        learning_rate=0.03,
        max_iter=500,
        min_samples_leaf=20,
        l2_regularization=2.0,
    ),
    dict(
        max_depth=5,
        learning_rate=0.03,
        max_iter=400,
        min_samples_leaf=10,
        l2_regularization=1.0,
    ),
]

results = []
for cfg in CONFIGS:
    s = time_cv(lambda: make_hgb(**cfg), FEATURES)
    results.append((cfg, s.mean()))
    print(
        f"depth={cfg['max_depth']} lr={cfg['learning_rate']:<5} iter={cfg['max_iter']:<4} "
        f"leaf={cfg['min_samples_leaf']:<3} l2={cfg['l2_regularization']:<4} -> "
        + " ".join(f"{x:.4f}" for x in s)
        + f"  среднее {s.mean():.4f}"
    )

BEST_CFG = max(results, key=lambda t: t[1])[0]
print("\nвыбрана конфигурация:", BEST_CFG)

depth=3 lr=0.05  iter=300  leaf=20  l2=1.0  -> 0.7168 0.7092 0.8175  среднее 0.7478
depth=3 lr=0.08  iter=250  leaf=30  l2=1.0  -> 0.6925 0.6920 0.7832  среднее 0.7226
depth=4 lr=0.05  iter=400  leaf=15  l2=1.0  -> 0.6850 0.7003 0.7724  среднее 0.7192
depth=4 lr=0.03  iter=500  leaf=20  l2=2.0  -> 0.6966 0.6944 0.7671  среднее 0.7194
depth=5 lr=0.03  iter=400  leaf=10  l2=1.0  -> 0.6729 0.6612 0.7635  среднее 0.6992

выбрана конфигурация: {'max_depth': 3, 'learning_rate': 0.05, 'max_iter': 300, 'min_samples_leaf': 20, 'l2_regularization': 1.0}


## 5. Важность признаков

Permutation importance на валидации с отсечкой 15 апреля.

In [11]:
from sklearn.inspection import permutation_importance

is_valid = window_start.ge("2026-04-15").values
model = make_hgb(**BEST_CFG)
model.fit(Xtr.loc[~is_valid, FEATURES], ytr[~is_valid])

perm = permutation_importance(
    model,
    Xtr.loc[is_valid, FEATURES],
    ytr[is_valid],
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(
    ascending=False
)
print(importance.head(15).round(4).to_string())

pointer_x_std        0.0940
iat_median           0.0514
category_nunique     0.0277
location_nunique     0.0167
iat_min              0.0155
item_revisit_frac    0.0120
item_nunique         0.0042
platform_nunique     0.0033
n_events             0.0031
search_page_mean     0.0028
item_view_nunique    0.0021
search_page_max      0.0015
span_sec             0.0014
cnt_eid_400          0.0014
cnt_eid_220          0.0013


Больше всего модель опирается на разброс курсора, интервалы между событиями,
разнообразие категорий/локаций и повторные обращения к объявлениям.

## 6. Сабмит

Финальная модель — на всём train.

In [12]:
final_model = make_hgb(**BEST_CFG)
final_model.fit(Xtr[FEATURES], ytr)

test_scores = final_model.predict_proba(Xte[FEATURES])[:, 1]

submission = pd.DataFrame(
    {
        "cookie_id": Xte["cookie_id"],
        "score": test_scores,
    }
)

assert len(submission) == len(test)
assert submission["cookie_id"].is_unique
assert set(submission["cookie_id"]) == set(test["cookie_id"])
assert submission["score"].between(0, 1).all()

submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())
print("\nраспределение score:")
print(submission["score"].describe())

(4909, 2)
             cookie_id     score
0  ck_315fb710a0e371e7  0.002056
1  ck_a76ee3b3e3e522fd  0.139121
2  ck_94c9a4d382689e82  0.045848
3  ck_8eaf9509ad9462a0  0.001147
4  ck_9a88a5a989cb5bc6  0.007106

распределение score:
count    4909.000000
mean        0.087529
std         0.205003
min         0.000284
25%         0.005586
50%         0.016104
75%         0.048819
max         0.995098
Name: score, dtype: float64


## 7. Описание подхода

**Данные.** Удалил дубликаты событий, считаю признаки только по событиям внутри окна.

**Признаки.** 35 агрегатов по событиям окна: объём и разнообразие действий, повторы
объявлений, ритм (интервалы между событиями), технические следы (платформы,
user-agent), разброс курсора, возраст куки.

**Валидация.** По времени, три точки отсечения (13, 15, 17 апреля).

**Модель.** `HistGradientBoostingClassifier` (sklearn). Средний P@R0.7 на валидации
≈ 0.75; baseline из quickstart ≈ 0.09, RandomForest ≈ 0.62, логрег ≈ 0.43.

**Ограничения.** Ботов в валидации немного (160–408), поэтому оценка шумная. Можно
попробовать разбирать браузер/ОС из user-agent и поисковые запросы.

**Воспроизводимость.** Только numpy, pandas, scikit-learn. `random_state=0`,
Python 3.14, версии в `requirements.txt`. Запускать из корня архива (рядом с
`metric.py` и `data/`).